# EdgarTools `to_markdown()` Demo

Demonstrates the new markdown output methods added to drill-down objects in the `markdown` branch.

**Features covered:**
- `create_markdown_table()` and `process_content()` from `edgar.markdown`
- `RenderedStatement.to_markdown()` with `detail` and `optimize_for_llm` params
- `Statement.to_markdown()`
- `StatementLineItem.to_markdown()` with note references and period labels
- `Note.to_markdown()` with tables rendered as pipe tables
- `Notes.to_markdown()` with focus filtering
- `TenK.to_context(focus=..., output_format='markdown')`

## 1. Low-Level Formatting Utilities

These are the building blocks ported from `quant/markdown/helpers.py`.

In [ ]:
from edgar.markdown import create_markdown_table, process_content

# create_markdown_table: headers + rows + optional alignment
md = create_markdown_table(
    headers=['Metric', 'FY 2024', 'FY 2023'],
    rows=[
        ['Revenue', '$394B', '$383B'],
        ['Net Income', '$97B', '$94B'],
        ['EPS (diluted)', '$6.42', '$6.13'],
    ],
    alignments=['left', 'right', 'right'],
)
print(md)

In [ ]:
# process_content: converts raw HTML (tables, headings, text) to LLM-optimized markdown
html = """
<h3>Debt Maturity Schedule</h3>
<table>
  <tr><th></th><th>2025</th><th>2026</th><th>2027</th><th>2028</th></tr>
  <tr><td>Term Debt</td><td>$10,500</td><td>$9,750</td><td>$8,250</td><td>$7,000</td></tr>
  <tr><td>Commercial Paper</td><td>$5,900</td><td>$4,800</td><td>$3,200</td><td>$2,100</td></tr>
  <tr><td><b>Total</b></td><td><b>$16,400</b></td><td><b>$14,550</b></td><td><b>$11,450</b></td><td><b>$9,100</b></td></tr>
</table>
<p>The Company issues unsecured short-term promissory notes under a commercial paper program.</p>
"""
print(process_content(html))

## 2. RenderedStatement.to_markdown()

Upgraded with company header, NBSP indentation, Rich tag stripping, and detail levels.

In [ ]:
from edgar.xbrl.rendering import RenderedStatement, StatementRow, StatementCell, StatementHeader

def cell(value, formatted=None):
    fmt = formatted or (f'{value:,.0f}' if isinstance(value, (int, float)) and value else '')
    return StatementCell(value=value, formatter=lambda v, f=fmt: f if v is not None else '')

rows = [
    StatementRow(label='Net sales', level=0, is_abstract=True,
                 cells=[cell(391035), cell(383285)], metadata={'concept': 'Revenue'}),
    StatementRow(label='Products', level=1,
                 cells=[cell(224578), cell(220272)], metadata={}),
    StatementRow(label='Services', level=1,
                 cells=[cell(166457), cell(163013)], metadata={}),
    StatementRow(label='Cost of sales', level=0,
                 cells=[cell(210352), cell(214137)], metadata={}),
    StatementRow(label='Gross profit', level=0, is_abstract=True,
                 cells=[cell(180683), cell(169148)], metadata={}),
]

rs = RenderedStatement(
    title='Income Statement',
    header=StatementHeader(columns=['2024-09-28', '2023-09-30']),
    rows=rows,
    metadata={'company_name': 'Apple Inc.', 'ticker': 'AAPL'},
    statement_type='IncomeStatement',
    fiscal_period_indicator='FY 2024',
    units_note='[italic]In millions[/italic]',
)

In [ ]:
# Standard detail (default)
print(rs.to_markdown())

In [ ]:
# Minimal — just the pipe table, no header
print(rs.to_markdown(detail='minimal'))

In [ ]:
# Full — header + footer with source attribution
print(rs.to_markdown(detail='full'))

In [ ]:
# optimize_for_llm: skips abstract rows that have no values
from edgar.xbrl.rendering import StatementRow, StatementCell

rows_with_empty_abstract = [
    StatementRow(label='ASSETS', level=0, is_abstract=True,
                 cells=[cell(None), cell(None)], metadata={}),
    StatementRow(label='Cash and equivalents', level=1,
                 cells=[cell(29965), cell(29965)], metadata={}),
    StatementRow(label='Short-term investments', level=1,
                 cells=[cell(31590), cell(27699)], metadata={}),
]

rs2 = RenderedStatement(
    title='Balance Sheet', header=StatementHeader(columns=['2024-09-28', '2023-09-30']),
    rows=rows_with_empty_abstract,
    metadata={'company_name': 'Apple Inc.', 'ticker': 'AAPL'},
    statement_type='BalanceSheet',
)

print('=== Normal ===')
print(rs2.to_markdown(detail='minimal'))
print()
print('=== LLM Optimized (ASSETS header dropped) ===')
print(rs2.to_markdown(detail='minimal', optimize_for_llm=True))

## 3. Statement.to_markdown() — Convenience Wrapper

Calls `self.render().to_markdown()` under the hood.

In [ ]:
# This requires a real Filing + XBRL parse, so we use a live example.
# If you don't have network access, skip this cell.

from edgar import Company

company = Company("MSFT")
filing = company.get_filings(form="10-K").latest()
financials = filing.xbrl().statements
income = financials.income_statement()

print(income.to_markdown())

In [ ]:
# Minimal detail — just the pipe table
print(income.to_markdown(detail='minimal'))

## 4. StatementLineItem.to_markdown()

Compact markdown for a single line item with values and optional note link.

In [ ]:
# Requires network — uses MSFT 10-K
from edgar import Company

company = Company("MSFT")
tenk = company.get_filings(form="10-K").latest().obj()
bs = tenk.financials.balance_sheet

# Look up a line item
item = bs['Goodwill']
if item:
    print('=== With note reference ===')
    print(item.to_markdown())
    print()
    print('=== Without note reference ===')
    print(item.to_markdown(include_note=False))
else:
    print('Goodwill not found in balance sheet — try searching:')
    for match in bs.search('goodwill'):
        print(f'  {match}')

## 5. Note.to_markdown()

Full markdown output for a single note — tables rendered as pipe tables via `process_content()`.

In [ ]:
# Requires network — uses MSFT 10-K
from edgar import Company

company = Company("MSFT")
tenk = company.get_filings(form="10-K").latest().obj()
notes = tenk.notes

# Find a note about debt
debt_notes = notes.search('debt')
if debt_notes:
    note = debt_notes[0]
    print(f'Found: {note}\n')
    print(note.to_markdown(detail='standard'))
else:
    print('No debt note found. Available notes:')
    for n in notes:
        print(f'  {n}')

In [ ]:
# Minimal detail — just the title and table names
if debt_notes:
    print(debt_notes[0].to_markdown(detail='minimal'))

In [ ]:
# Full detail — includes policies and detail breakdowns
if debt_notes:
    print(debt_notes[0].to_markdown(detail='full'))

## 6. Notes.to_markdown()

Render all notes (or a focused subset) as a single markdown document.

In [ ]:
# Full document — all notes (minimal detail to keep output manageable)
print(notes.to_markdown(detail='minimal'))

In [ ]:
# Focused on specific topics
print(notes.to_markdown(detail='standard', focus='revenue'))

In [ ]:
# Multi-topic focus
print(notes.to_markdown(detail='standard', focus=['debt', 'revenue']))

## 7. TenK.to_context(focus=..., output_format='markdown')

The `output_format='markdown'` param routes through `Notes.to_markdown()` for GFM output.

In [ ]:
# Text format (default — backward compatible)
print('=== TEXT FORMAT ===')
print(tenk.to_context(focus='debt', detail='minimal'))
print()
print('=== MARKDOWN FORMAT ===')
print(tenk.to_context(focus='debt', detail='minimal', output_format='markdown'))

In [ ]:
# Full markdown with multiple topics
print(tenk.to_context(focus=['debt', 'goodwill'], detail='standard', output_format='markdown'))

## 8. Side-by-Side Comparison: text vs markdown

Compare the old `to_context()` plain text with the new markdown output.

In [ ]:
if debt_notes:
    note = debt_notes[0]
    print('=' * 60)
    print('to_context() — plain text')
    print('=' * 60)
    print(note.to_context(detail='standard'))
    print()
    print('=' * 60)
    print('to_markdown() — GFM with pipe tables')
    print('=' * 60)
    print(note.to_markdown(detail='standard'))

## 9. Save Markdown to File

Write the output to a `.md` file and open in any markdown viewer.

In [ ]:
from pathlib import Path

# Save full notes document
output = notes.to_markdown(detail='standard', focus=['debt', 'revenue'])
output_path = Path('demo_output.md')
output_path.write_text(output, encoding='utf-8')
print(f'Saved {len(output):,} characters to {output_path.resolve()}')
print(f'Open in VS Code: code {output_path}')